In [ ]:
## https://chatgpt.com/c/6838502d-0b7c-8006-860f-0fc463bfc059

In [ ]:
pip install fuzzywuzzy

In [ ]:
pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.0 MB/s eta 0:00:00


In [ ]:
pip install fuzzywuzzy python-Levenshtein requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from fuzzywuzzy import process, fuzz
import re

# Chargement du catalogue de voitures
cars_db = pd.read_csv('cars2025.csv')

# Chargement des noms erronés
errors_db = pd.read_csv('cars_error3.csv')

# Prétraitement amélioré avec gestion des caractères spéciaux
def preprocess_text(text):
    """Nettoyage et normalisation du texte"""
    if not isinstance(text, str):
        return ""

    # Normalisation de base
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)  # Supprime la ponctuation
    text = re.sub(r'\s+', ' ', text)      # Supprime les espaces multiples

    # Correction d'abréviations courantes
    abbr_dict = {
        'bw': 'bmw',
        'vw': 'volkswagen',
        'v w': 'volkswagen',
        'accrd': 'accord',
        'ferari': 'ferrari',
        'toy': 'toyota',
        'ren': 'renault',
        'peuj': 'peugeot',
        'cit': 'citroën',
        'merced': 'mercedes'
    }

    for abbr, full in abbr_dict.items():
        text = re.sub(rf'\b{abbr}\b', full, text)

    return text

# Création de la liste des noms complets
cars_db['full_name'] = cars_db['Marque'] + ' ' + cars_db['Modèle']
full_names = [preprocess_text(name) for name in cars_db['full_name'].tolist()]

# Dictionnaire pour recherche rapide par marque
brands_dict = {}
for brand in cars_db['Marque'].unique():
    cleaned_brand = preprocess_text(brand)
    if cleaned_brand not in brands_dict:
        brands_dict[cleaned_brand] = []
    brand_models = cars_db[cars_db['Marque'] == brand]['Modèle'].tolist()
    brands_dict[cleaned_brand] = [preprocess_text(m) for m in brand_models]

# Fonction de correction optimisée
def correct_car_name(input_name, threshold=75):
    """Corrige le nom de voiture avec correspondance floue"""
    cleaned_input = preprocess_text(input_name)

    # 1. Recherche exacte
    if cleaned_input in full_names:
        idx = full_names.index(cleaned_input)
        return cars_db.iloc[idx]['full_name'], 100

    # 2. Recherche floue globale
    matches = process.extractBests(
        cleaned_input,
        full_names,
        scorer=fuzz.token_set_ratio,
        score_cutoff=threshold-20,
        limit=5
    )

    if matches:
        best_match, score = max(matches, key=lambda x: x[1])
        if score >= threshold:
            idx = full_names.index(best_match)
            return cars_db.iloc[idx]['full_name'], score

    # 3. Recherche par marque et modèle
    words = cleaned_input.split()

    # Cas des entrées très courtes (ex: "BW")
    if len(words) == 1 and len(cleaned_input) <= 4:
        brand_match, brand_score = process.extractOne(cleaned_input, brands_dict.keys())
        if brand_score > 80:
            # Retourne le modèle le plus courant de la marque
            popular_model = cars_db[cars_db['Marque'].str.lower() == brand_match]['Modèle'].value_counts().index[0]
            return f"{brand_match.title()} {popular_model}", brand_score

    # Cas normal avec plusieurs mots
    if len(words) >= 2:
        # Recherche de la marque
        brand_match, brand_score = process.extractOne(words[0], brands_dict.keys())

        # Règles spéciales pour marques connues
        if brand_score < 70:
            if 'ferrari' in cleaned_input or 'ferari' in cleaned_input:
                brand_match = 'ferrari'
                brand_score = 95
            elif 'merced' in cleaned_input:
                brand_match = 'mercedes'
                brand_score = 90

        if brand_score > 65:
            # Recherche du modèle
            model_text = ' '.join(words[1:])
            model_match, model_score = process.extractOne(
                model_text,
                brands_dict[brand_match],
                scorer=fuzz.partial_ratio
            )

            # Règles spéciales pour modèles courts
            if model_score < 80 and len(model_text) <= 3:
                for model in brands_dict[brand_match]:
                    if model_text in model:
                        model_match = model
                        model_score = 85
                        break

            if model_score > 60:
                return f"{brand_match.title()} {model_match.title()}", (brand_score + model_score)//2

    # 4. Dernière tentative avec le meilleur match
    if matches:
        best_match, score = matches[0]
        idx = full_names.index(best_match)
        return cars_db.iloc[idx]['full_name'], score

    return "non reconnue", 0

# Traitement des erreurs
results = []
for idx, row in errors_db.iterrows():
    erroneous_name = row['Nom_Véhicule']
    corrected, confidence = correct_car_name(erroneous_name)
    results.append({
        'Nom_Erroné': erroneous_name,
        'Correction': corrected,
        'Confiance (%)': confidence
    })

# Post-traitement pour forcer les corrections connues
correction_rules = {
    r'tota.*co': 'Toyota Corolla',
    r'^bw$': 'BMW Série 3',
    r'ferari': 'Ferrari',
    r'honda.*accrd': 'Honda Accord',
    r'peugeot.*208': 'Peugeot 208',
    r'renault.*clio': 'Renault Clio',
    r'volkswagen.*golf': 'Volkswagen Golf'
}

for result in results:
    for pattern, replacement in correction_rules.items():
        if re.search(pattern, result['Nom_Erroné'], re.IGNORECASE):
            result['Correction'] = replacement
            result['Confiance (%)'] = 95
            break

# Création du DataFrame et sauvegarde
results_df = pd.DataFrame(results)
results_df.to_csv('corrections_finales.csv', index=False)

print("Corrections terminées !")
print(f"{len(results_df)} entrées traitées")
print("\nAperçu des résultats :")
print(results_df.head())

Corrections terminées !
20 entrées traitées

Aperçu des résultats :
    Nom_Erroné      Correction  Confiance (%)
0      Tota Co  Toyota Corolla             95
1           BW     BMW Série 3             95
2  Peugeot 208     Peugeot 208             95
3       Ferari         Ferrari             95
4  Honda Accrd    Honda Accord             95
